# Sitzung 4: Extremwertprobleme & Steckbriefaufgaben — Interaktiv

In diesem Notebook kannst du:
1. **Dose optimieren** — Slider für Radius, Live-Visualisierung von Oberfläche und Volumen
2. **Zaun-Problem** erkunden — Rechteck an einer Mauer, maximale Fläche finden
3. **Allgemeiner Extremwert-Löser** — Zielfunktion + Nebenbedingung mit sympy
4. **Steckbriefaufgaben** interaktiv lösen — Bedingungen eingeben, Funktion berechnen
5. **Methode Schritt für Schritt** — Vom Sachproblem zur Lösung

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from ipywidgets import interact, FloatSlider, Dropdown, Text, IntSlider
from IPython.display import display, Markdown, HTML
import sympy as sp
%matplotlib inline

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['text.usetex'] = False
plt.rcParams['mathtext.fontset'] = 'cm'  # Computer Modern (LaTeX-Look)

---
## 1. Dose optimieren — Minimale Oberfläche bei festem Volumen

Eine zylindrische Dose soll ein Volumen von **500 cm³** haben. Welcher Radius minimiert den Materialverbrauch (= Oberfläche)?

**Nebenbedingung:** $V = \pi r^2 h = 500$ → $h = \frac{500}{\pi r^2}$

**Zielfunktion:** $O(r) = 2\pi r^2 + 2\pi r h = 2\pi r^2 + \frac{1000}{r}$

Verschiebe den Slider und beobachte, wie sich Dose und Oberfläche ändern.

In [ ]:
V_DOSE = 500  # cm³

def dose_explorer(r=4.3):
    h = V_DOSE / (np.pi * r**2)
    O_aktuell = 2 * np.pi * r**2 + 2 * np.pi * r * h

    # Optimaler Radius
    r_opt = (V_DOSE / (2 * np.pi)) ** (1/3)
    h_opt = V_DOSE / (np.pi * r_opt**2)
    O_opt = 2 * np.pi * r_opt**2 + 2 * np.pi * r_opt * h_opt

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    # --- Links: Dose als Seitenansicht ---
    dose_breite = 2 * r
    max_dim = 25
    ax1.set_xlim(-max_dim/2, max_dim/2)
    ax1.set_ylim(0, max_dim)
    rect = patches.FancyBboxPatch((-r, 0.5), dose_breite, h,
                                   boxstyle="round,pad=0.3",
                                   facecolor='#4A90D9', alpha=0.6,
                                   edgecolor='#2C5F8A', linewidth=2)
    ax1.add_patch(rect)
    ax1.set_aspect('equal')
    ax1.set_title(f'Dose: $r = {r:.1f}$ cm, $h = {h:.1f}$ cm', fontsize=13)
    ax1.annotate(f'$O = {O_aktuell:.0f}$ cm²', xy=(0, h/2 + 0.5),
                fontsize=14, ha='center', va='center',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    # Bemaßung
    ax1.annotate('', xy=(r + 1, 0.5), xytext=(r + 1, h + 0.5),
                arrowprops=dict(arrowstyle='<->', color='red', lw=1.5))
    ax1.text(r + 2, (h + 1) / 2, f'$h={h:.1f}$', color='red', fontsize=11, va='center')
    ax1.annotate('', xy=(-r, -1), xytext=(r, -1),
                arrowprops=dict(arrowstyle='<->', color='green', lw=1.5))
    ax1.text(0, -2.5, f'$2r={2*r:.1f}$', color='green', fontsize=11, ha='center')
    ax1.axis('off')

    # --- Rechts: O(r)-Plot ---
    rs = np.linspace(1, 12, 300)
    Os = 2 * np.pi * rs**2 + 2 * V_DOSE / rs
    ax2.plot(rs, Os, 'b-', linewidth=2, label=r'$O(r) = 2\pi r^2 + \frac{1000}{r}$')
    ax2.plot(r_opt, O_opt, 'g*', markersize=18, zorder=5,
             label=rf'Optimum: $r^* = {r_opt:.2f}$ cm')
    ax2.plot(r, O_aktuell, 'ro', markersize=12, zorder=5,
             label=rf'Aktuell: $r = {r:.1f}$ cm')
    ax2.axvline(x=r, color='red', linestyle=':', alpha=0.5)
    ax2.set_xlabel('Radius $r$ [cm]', fontsize=12)
    ax2.set_ylabel('Oberfläche $O$ [cm²]', fontsize=12)
    ax2.set_ylim(300, 1200)
    ax2.grid(True, alpha=0.3)
    ax2.legend(fontsize=11, loc='upper right', framealpha=0.9)
    ax2.set_title('Oberfläche in Abhängigkeit vom Radius', fontsize=13)

    plt.tight_layout()
    plt.show()

interact(
    dose_explorer,
    r=FloatSlider(value=4.3, min=1.5, max=10.0, step=0.1,
                  description='Radius r:', readout_format='.1f',
                  style={'description_width': '80px'}, layout={'width': '450px'})
);

**Beobachte:**
- Bei kleinem Radius ist die Dose sehr hoch → viel Mantelfläche
- Bei großem Radius sind die Deckel riesig → viel Deckelfläche
- Das **Optimum** liegt dazwischen, wo $O'(r) = 0$
- Beim optimalen Zylinder gilt: **Höhe = Durchmesser** ($h = 2r$)

---
## 2. Zaun-Problem — Maximale Fläche an einer Mauer

Ein Rechteck wird an eine Mauer gebaut. Verfügbarer Zaun: **40 m** (nur 3 Seiten).

**Nebenbedingung:** $2x + y = 40$ → $y = 40 - 2x$

**Zielfunktion:** $A(x) = x \cdot y = x(40 - 2x) = 40x - 2x^2$

In [ ]:
L_ZAUN = 40  # Meter verfügbarer Zaun

def zaun_explorer(x=10.0):
    y = L_ZAUN - 2 * x
    A_aktuell = x * y
    x_opt = L_ZAUN / 4
    A_opt = x_opt * (L_ZAUN - 2 * x_opt)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    # --- Links: Gehege-Skizze ---
    ax1.set_xlim(-2, 25)
    ax1.set_ylim(-3, 18)

    # Mauer
    ax1.fill_between([0, 22], [15, 15], [16, 16], color='#8B7355', alpha=0.8)
    ax1.text(11, 15.5, 'MAUER', ha='center', va='center', fontsize=11,
             fontweight='bold', color='white')

    # Gehege
    if y > 0:
        gehege = patches.Rectangle((1, 15 - x), y, x,
                                    facecolor='#90EE90', alpha=0.4,
                                    edgecolor='#228B22', linewidth=2.5)
        ax1.add_patch(gehege)

        # Bemaßung
        ax1.annotate('', xy=(-0.5, 15), xytext=(-0.5, 15 - x),
                    arrowprops=dict(arrowstyle='<->', color='red', lw=2))
        ax1.text(-1.5, 15 - x/2, f'$x={x:.0f}$', color='red', fontsize=12,
                ha='center', va='center')
        ax1.annotate('', xy=(1, 15 - x - 1), xytext=(1 + y, 15 - x - 1),
                    arrowprops=dict(arrowstyle='<->', color='blue', lw=2))
        ax1.text(1 + y/2, 15 - x - 2, f'$y={y:.0f}$', color='blue', fontsize=12,
                ha='center')
        ax1.text(1 + y/2, 15 - x/2, f'$A = {A_aktuell:.0f}$ m²',
                fontsize=14, ha='center', va='center',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    ax1.set_title(f'Gehege: $x = {x:.0f}$ m, $y = {y:.0f}$ m', fontsize=13)
    ax1.axis('off')

    # --- Rechts: A(x)-Plot ---
    xs = np.linspace(0, L_ZAUN / 2, 300)
    As = xs * (L_ZAUN - 2 * xs)
    ax2.plot(xs, As, 'b-', linewidth=2, label=r'$A(x) = x(40 - 2x)$')
    ax2.plot(x_opt, A_opt, 'g*', markersize=18, zorder=5,
             label=rf'Optimum: $x^* = {x_opt:.0f}$ m, $A = {A_opt:.0f}$ m²')
    if 0 < x < L_ZAUN / 2:
        ax2.plot(x, A_aktuell, 'ro', markersize=12, zorder=5,
                 label=rf'Aktuell: $x = {x:.0f}$ m')
        ax2.axvline(x=x, color='red', linestyle=':', alpha=0.5)
    ax2.set_xlabel('Seitenlänge $x$ [m]', fontsize=12)
    ax2.set_ylabel('Fläche $A$ [m²]', fontsize=12)
    ax2.set_ylim(0, 250)
    ax2.grid(True, alpha=0.3)
    ax2.legend(fontsize=11, loc='upper left', framealpha=0.9)
    ax2.set_title('Fläche in Abhängigkeit von $x$', fontsize=13)

    plt.tight_layout()
    plt.show()

interact(
    zaun_explorer,
    x=FloatSlider(value=10, min=1, max=19, step=1,
                  description='Seite x:', readout_format='.0f',
                  style={'description_width': '70px'}, layout={'width': '450px'})
);

**Beobachte:**
- $x = 0$ oder $x = 20$: Fläche = 0 (degeneriertes Rechteck)
- **Maximum** bei $x = 10$ m, $y = 20$ m → $A = 200$ m²
- Die Zielfunktion ist eine nach unten geöffnete Parabel — das Maximum liegt am Scheitel

---
## 2b. Blech-Quader — Maximales Volumen

Aus einem quadratischen Blech (20 × 20 cm) werden an den Ecken Quadrate mit Seitenlänge $x$ ausgeschnitten und die Ränder hochgeklappt → offener Quader.

**Zielfunktion:** $V(x) = x \cdot (20 - 2x)^2$

**Definitionsbereich:** $0 < x < 10$

In [ ]:
A_BLECH = 20  # Seitenlänge des Blechs in cm

def blech_explorer(x=3.3):
    seite = A_BLECH - 2 * x
    V_aktuell = x * seite**2

    x_opt = A_BLECH / 6
    V_opt = x_opt * (A_BLECH - 2 * x_opt)**2

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))

    # --- Links: Blech-Aufsicht mit Ausschnitten und gefaltetem Quader ---
    ax1.set_xlim(-1, 22)
    ax1.set_ylim(-3, 22)
    ax1.set_aspect('equal')

    # Blech-Grundfläche (grau)
    blech = patches.Rectangle((0, 0), A_BLECH, A_BLECH,
                                facecolor='#D3D3D3', edgecolor='#555', linewidth=1.5)
    ax1.add_patch(blech)

    # Ausgeschnittene Ecken (weiß)
    for cx, cy in [(0, 0), (A_BLECH - x, 0), (0, A_BLECH - x), (A_BLECH - x, A_BLECH - x)]:
        ecke = patches.Rectangle((cx, cy), x, x,
                                  facecolor='white', edgecolor='red',
                                  linewidth=2, linestyle='--')
        ax1.add_patch(ecke)

    # Boden hervorheben (blau)
    boden = patches.Rectangle((x, x), seite, seite,
                                facecolor='#4A90D9', alpha=0.3,
                                edgecolor='#2C5F8A', linewidth=2)
    ax1.add_patch(boden)

    # Bemaßung x
    ax1.annotate('', xy=(0, -1), xytext=(x, -1),
                arrowprops=dict(arrowstyle='<->', color='red', lw=1.5))
    ax1.text(x / 2, -2, f'$x={x:.1f}$', color='red', fontsize=11, ha='center')

    # Bemaßung Seite
    ax1.annotate('', xy=(x, -1), xytext=(A_BLECH - x, -1),
                arrowprops=dict(arrowstyle='<->', color='blue', lw=1.5))
    ax1.text(A_BLECH / 2, -2, f'${seite:.1f}$', color='blue', fontsize=11, ha='center')

    ax1.text(A_BLECH / 2, A_BLECH / 2, f'Boden\n${seite:.1f} \\times {seite:.1f}$\nHöhe $= {x:.1f}$',
            fontsize=12, ha='center', va='center',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    ax1.text(A_BLECH / 2, A_BLECH + 0.8,
             f'$V = {x:.1f} \\cdot {seite:.1f}^2 = {V_aktuell:.0f}$ cm³',
             fontsize=13, ha='center', fontweight='bold',
             bbox=dict(boxstyle='round', facecolor='#FFF3E0', alpha=0.9))

    ax1.set_title('Blech-Aufsicht (rot = Ausschnitte)', fontsize=13)
    ax1.axis('off')

    # --- Rechts: V(x)-Plot ---
    xs = np.linspace(0, A_BLECH / 2, 300)
    Vs = xs * (A_BLECH - 2 * xs)**2

    ax2.plot(xs, Vs, 'b-', linewidth=2.5,
             label=r'$V(x) = x \cdot (20-2x)^2$')
    ax2.plot(x_opt, V_opt, 'g*', markersize=18, zorder=5,
             label=rf'Optimum: $x^* = \frac{{10}}{{3}} \approx {x_opt:.2f}$ cm')
    if 0 < x < A_BLECH / 2:
        ax2.plot(x, V_aktuell, 'ro', markersize=12, zorder=5,
                 label=rf'Aktuell: $x = {x:.1f}$ cm')
        ax2.axvline(x=x, color='red', linestyle=':', alpha=0.5)

    ax2.set_xlabel('Ausschnitt $x$ [cm]', fontsize=12)
    ax2.set_ylabel('Volumen $V$ [cm³]', fontsize=12)
    ax2.set_xlim(0, 10)
    ax2.set_ylim(0, 700)
    ax2.grid(True, alpha=0.3)
    ax2.legend(fontsize=11, loc='upper right', framealpha=0.9)
    ax2.set_title('Volumen in Abhängigkeit von $x$', fontsize=13)

    plt.tight_layout()
    plt.show()

interact(
    blech_explorer,
    x=FloatSlider(value=3.3, min=0.5, max=9.5, step=0.1,
                  description='Ausschnitt x:', readout_format='.1f',
                  style={'description_width': '100px'}, layout={'width': '450px'})
);

**Beobachte:**
- Bei $x \to 0$: Blech fast nicht gefaltet → kein Volumen
- Bei $x \to 10$: Seitenlänge wird 0 → kein Volumen
- **Optimum** bei $x = \frac{10}{3} \approx 3{,}33$ cm → $V_{\max} \approx 593$ cm³
- $V'(x) = (20-2x)(20-6x) = 0$ liefert $x = 10$ (trivial) und $x = \frac{10}{3}$ (Maximum)

---
## 2c. Jetzt bist du dran! — Extremwertprobleme selbst aufstellen

Der **schwierigste Schritt** bei Extremwertaufgaben ist das Modellieren: Zielfunktion und Nebenbedingung richtig aufstellen. Übe das hier!

Trage in den Zellen unten die fehlenden Ausdrücke ein und führe die Zelle aus. Wenn alles stimmt, berechnet der Löser das Ergebnis.

### Aufgabe A1: Draht-Rechteck (60 cm Draht → maximale Fläche)

Variablen: $x$ = eine Seite, $y$ = andere Seite.
- **Nebenbedingung:** Der Umfang ist 60 cm → $2x + 2y = 60$
- **Zielfunktion:** Die Fläche $A = x \cdot y$ soll maximal werden

Ersetze die `'...'` durch die richtigen Ausdrücke:

In [ ]:
# ✏️ Aufgabe A1: Draht-Rechteck
# Trage die Zielfunktion und Nebenbedingung ein!

extremwert_loesen(
    ziel_str='...',           # Was wird maximiert? (Fläche = x * y)
    neben_str='... = ...',    # Umfang = 60
    variable_str='y'          # y wird eliminiert
)

### Aufgabe A3: Zaun an der Mauer (40 m Zaun → maximale Fläche)

Variablen: $x$ = Seite senkrecht zur Mauer, $y$ = Seite parallel zur Mauer.
- **Nebenbedingung:** Nur 3 Seiten brauchen Zaun → $2x + y = 40$
- **Zielfunktion:** $A = x \cdot y$

Ersetze die `'...'`:

In [ ]:
# ✏️ Aufgabe A3: Zaun an der Mauer
# Trage die Zielfunktion und Nebenbedingung ein!

extremwert_loesen(
    ziel_str='...',           # Was wird maximiert?
    neben_str='... = ...',    # Nebenbedingung (Zaunlänge)
    variable_str='y'          # y wird eliminiert
)

In [ ]:
_x, _y = sp.symbols('x y', positive=True)

def extremwert_loesen(ziel_str, neben_str, variable_str):
    """
    Löst ein Extremwertproblem.

    Parameter:
        ziel_str:     Zielfunktion als String, z.B. '2*pi*x**2 + 2*pi*x*y'
        neben_str:    Nebenbedingung '... = ...', z.B. 'pi*x**2*y = 500'
        variable_str: Variable, die eliminiert wird, z.B. 'y'
    """
    lokale = {'x': _x, 'y': _y, 'pi': sp.pi, 'sqrt': sp.sqrt,
              'exp': sp.exp, 'ln': sp.log, 'log': sp.log}

    ziel = sp.sympify(ziel_str.replace('^', '**'), locals=lokale)
    # Nebenbedingung parsen: 'lhs = rhs'
    teile = neben_str.replace('^', '**').split('=')
    if len(teile) != 2:
        print("Fehler: Nebenbedingung muss die Form 'lhs = rhs' haben.")
        return
    lhs = sp.sympify(teile[0].strip(), locals=lokale)
    rhs = sp.sympify(teile[1].strip(), locals=lokale)
    nebenbedingung = sp.Eq(lhs, rhs)

    elim_var = lokale.get(variable_str.strip(), sp.Symbol(variable_str.strip()))

    display(Markdown("### Schritt 1: Nebenbedingung nach $" +
                     sp.latex(elim_var) + "$ auflösen"))
    loesung_neben = sp.solve(nebenbedingung, elim_var)
    if not loesung_neben:
        print("Keine Lösung für die Nebenbedingung gefunden.")
        return
    display(Markdown(f"$${sp.latex(elim_var)} = {sp.latex(loesung_neben[0])}$$"))

    display(Markdown("### Schritt 2: In Zielfunktion einsetzen"))
    ziel_eingesetzt = ziel.subs(elim_var, loesung_neben[0])
    ziel_vereinfacht = sp.simplify(ziel_eingesetzt)
    # Bestimme die verbleibende Variable
    freie_vars = ziel_vereinfacht.free_symbols
    if not freie_vars:
        print("Nach Einsetzen keine freie Variable mehr.")
        return
    opt_var = list(freie_vars)[0]
    display(Markdown(f"$$Z({sp.latex(opt_var)}) = {sp.latex(ziel_vereinfacht)}$$"))

    display(Markdown("### Schritt 3: Ableiten und Nullstelle finden"))
    ableitung = sp.diff(ziel_vereinfacht, opt_var)
    display(Markdown(f"$$Z'({sp.latex(opt_var)}) = {sp.latex(sp.simplify(ableitung))}$$"))
    kritische = sp.solve(ableitung, opt_var)
    # Nur positive reelle Lösungen
    kritische_positiv = [k for k in kritische
                         if k.is_real is not False and k.is_positive is not False]
    if not kritische_positiv:
        print("Keine positive kritische Stelle gefunden.")
        return

    display(Markdown("### Schritt 4: Ergebnis"))
    for k in kritische_positiv:
        z2 = sp.diff(ableitung, opt_var).subs(opt_var, k)
        art = "Minimum" if z2 > 0 else ("Maximum" if z2 < 0 else "unbestimmt")
        elim_wert = loesung_neben[0].subs(opt_var, k)
        ziel_wert = ziel_vereinfacht.subs(opt_var, k)
        display(Markdown(
            f"- ${sp.latex(opt_var)}^* = {sp.latex(sp.simplify(k))} "
            f"\\approx {float(k):.4f}$\n"
            f"- ${sp.latex(elim_var)}^* = {sp.latex(sp.simplify(elim_wert))} "
            f"\\approx {float(elim_wert):.4f}$\n"
            f"- $Z^* = {sp.latex(sp.simplify(ziel_wert))} "
            f"\\approx {float(ziel_wert):.4f}$\n"
            f"- **Art:** {art} ($Z'' > 0$)" if art == "Minimum" else
            f"- ${sp.latex(opt_var)}^* = {sp.latex(sp.simplify(k))} "
            f"\\approx {float(k):.4f}$\n"
            f"- ${sp.latex(elim_var)}^* = {sp.latex(sp.simplify(elim_wert))} "
            f"\\approx {float(elim_wert):.4f}$\n"
            f"- $Z^* = {sp.latex(sp.simplify(ziel_wert))} "
            f"\\approx {float(ziel_wert):.4f}$\n"
            f"- **Art:** {art}"
        ))

# Beispiel: Dose
print("═" * 50)
print("Beispiel: Zylindrische Dose (V = 500 cm³)")
print("═" * 50)
extremwert_loesen(
    ziel_str='2*pi*x**2 + 2*pi*x*y',
    neben_str='pi*x**2*y = 500',
    variable_str='y'
)

### Lösungen zum Vergleich

Hast du die Aufgaben oben gelöst? Hier die Eingaben zur Kontrolle:

| Problem | `ziel_str` | `neben_str` | `variable_str` |
|---------|-----------|-------------|----------------|
| A1: Draht-Rechteck | `'x * y'` | `'2*x + 2*y = 60'` | `'y'` |
| A3: Zaun (40 m) | `'x * y'` | `'2*x + y = 40'` | `'y'` |

---
## 3. Allgemeiner Extremwert-Löser mit sympy

Den Löser oben kannst du auch für beliebige andere Probleme verwenden. Ändere die Zelle unten:

In [ ]:
def steckbrief_loesen_interaktiv(grad, bedingungen):
    """
    Löst eine Steckbriefaufgabe und zeigt alle Schritte.

    Parameter:
        grad: Grad des Polynoms (int)
        bedingungen: Liste von (typ, x_wert, y_wert)
            typ: 0 = f(x)=y, 1 = f'(x)=y, 2 = f''(x)=y
    """
    n_koeffs = grad + 1
    n_bed = len(bedingungen)

    # --- Eingabevalidierung ---
    if n_bed < n_koeffs:
        display(Markdown(
            f"**⚠ Zu wenige Bedingungen!** Ein Polynom {grad}. Grades hat "
            f"**{n_koeffs} Koeffizienten**, aber du hast nur **{n_bed} Bedingungen** "
            f"angegeben. Es fehlen noch **{n_koeffs - n_bed}**."
        ))
        return None
    if n_bed > n_koeffs:
        display(Markdown(
            f"**⚠ Zu viele Bedingungen!** Ein Polynom {grad}. Grades hat "
            f"**{n_koeffs} Koeffizienten**, aber du hast **{n_bed} Bedingungen** "
            f"angegeben. Das System ist überbestimmt — prüfe, ob der Grad stimmt."
        ))

    x = sp.Symbol('x')
    koeffs = sp.symbols(f'a0:{grad + 1}')
    poly = sum(koeffs[k] * x**k for k in range(grad + 1))
    poly_prime = sp.diff(poly, x)
    poly_pp = sp.diff(poly, x, 2)

    typ_namen = {0: 'f', 1: "f'", 2: "f''"}
    typ_ausdruecke = {0: poly, 1: poly_prime, 2: poly_pp}

    display(Markdown("### Ansatz"))
    display(Markdown(f"$$f(x) = {sp.latex(poly)}$$"))
    display(Markdown(f"$$f'(x) = {sp.latex(poly_prime)}$$"))
    if any(t == 2 for t, _, _ in bedingungen):
        display(Markdown(f"$$f''(x) = {sp.latex(poly_pp)}$$"))

    display(Markdown("### Gleichungssystem"))
    gleichungen = []
    for i, (typ, x_val, y_val) in enumerate(bedingungen):
        expr = typ_ausdruecke[typ]
        gl = sp.Eq(expr.subs(x, x_val), y_val)
        gleichungen.append(gl)
        lhs_vereinfacht = sp.simplify(expr.subs(x, x_val))
        display(Markdown(
            f"$({i+1})\\quad {typ_namen[typ]}({x_val}) = {y_val}"
            f"\\;\\Rightarrow\\; {sp.latex(lhs_vereinfacht)} = {y_val}$"
        ))

    display(Markdown("### Lösung"))
    loesung = sp.solve(gleichungen, koeffs)
    if not loesung:
        display(Markdown(
            "**❌ Keine Lösung gefunden!** Die Bedingungen widersprechen sich. "
            "Prüfe, ob alle Werte und Typen (f, f', f'') korrekt sind."
        ))
        return None

    for k in range(grad + 1):
        display(Markdown(f"$a_{k} = {sp.latex(loesung[koeffs[k]])}$"))

    f_ergebnis = poly.subs(loesung)
    f_vereinfacht = sp.expand(f_ergebnis)
    display(Markdown(f"### Ergebnis: $f(x) = {sp.latex(f_vereinfacht)}$"))

    # --- Plot mit dynamischen Achsen ---
    # x-Bereich aus den gegebenen Bedingungspunkten ableiten
    x_werte = [xv for _, xv, _ in bedingungen]
    y_werte_bed = [yv for t, _, yv in bedingungen if t == 0]  # nur f(x)=y Punkte

    x_min_bed = min(x_werte) if x_werte else -2
    x_max_bed = max(x_werte) if x_werte else 2
    x_span = max(x_max_bed - x_min_bed, 2)
    x_plot_min = x_min_bed - 0.5 * x_span
    x_plot_max = x_max_bed + 0.5 * x_span

    koeff_liste = [float(loesung[koeffs[k]]) for k in range(grad, -1, -1)]
    xs = np.linspace(x_plot_min, x_plot_max, 500)
    ys = np.polyval(koeff_liste, xs)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(xs, ys, 'b-', linewidth=2.5,
            label=rf'$f(x) = {sp.latex(f_vereinfacht)}$')

    # Bedingungspunkte einzeichnen
    for typ, x_val, y_val in bedingungen:
        if typ == 0:
            ax.plot(x_val, y_val, 'ro', markersize=10, zorder=5)
            ax.annotate(f'$({x_val}, {y_val})$', xy=(x_val, y_val),
                       xytext=(10, 10), textcoords='offset points', fontsize=11)
        elif typ == 1:
            # Tangente am Punkt andeuten
            f_val = float(np.polyval(koeff_liste, x_val))
            tang_dx = 0.5
            ax.plot([x_val - tang_dx, x_val + tang_dx],
                    [f_val - y_val * tang_dx, f_val + y_val * tang_dx],
                    'g--', linewidth=1.5, alpha=0.7)
            ax.plot(x_val, f_val, 'gs', markersize=8, zorder=5)

    ax.axhline(y=0, color='k', linewidth=0.5)
    ax.axvline(x=0, color='k', linewidth=0.5)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=13, loc='upper left', framealpha=0.9)
    ax.set_title('Rekonstruierte Funktion', fontsize=14)

    # y-Achse dynamisch anpassen
    ys_finite = ys[np.isfinite(ys)]
    if len(ys_finite) > 0:
        y_range = max(np.nanmax(np.abs(ys_finite)), 2)
        y_lo = min(np.nanmin(ys_finite) * 1.3, -1)
        y_hi = max(np.nanmax(ys_finite) * 1.3, 1)
        ax.set_ylim(y_lo, y_hi)

    plt.tight_layout()
    plt.show()

    return koeff_liste

### Beispiel B1: $f(0)=2,\; f'(0)=0,\; f(2)=0,\; f'(2)=0$

In [ ]:
steckbrief_loesen_interaktiv(3, [
    (0, 0, 2),   # f(0) = 2
    (1, 0, 0),   # f'(0) = 0
    (0, 2, 0),   # f(2) = 0
    (1, 2, 0),   # f'(2) = 0
]);

### Beispiel B2: $f'(1)=0,\; f(1)=4,\; f(0)=3$

In [ ]:
steckbrief_loesen_interaktiv(2, [
    (1, 1, 0),   # f'(1) = 0
    (0, 1, 4),   # f(1) = 4
    (0, 0, 3),   # f(0) = 3
]);

### Eigene Steckbriefaufgabe ausprobieren

Ändere die Zelle unten, um eigene Bedingungen einzugeben.

**Format:** `(typ, x_wert, y_wert)` mit:
- `typ = 0` → $f(x) = y$
- `typ = 1` → $f'(x) = y$
- `typ = 2` → $f''(x) = y$

In [ ]:
# Hier eigene Aufgabe eingeben:
steckbrief_loesen_interaktiv(
    grad=3,
    bedingungen=[
        (0, 0, 0),    # f(0) = 0   (Graph geht durch Ursprung)
        (1, 0, 3),    # f'(0) = 3  (Steigung 3 im Ursprung)
        (2, 2, 0),    # f''(2) = 0 (Wendepunkt bei x=2)
        (0, 2, 6),    # f(2) = 6   (Funktionswert am Wendepunkt)
    ]
)
# Überraschung: Die Bedingungen erzwingen f(x) = 3x — gar kein Kubik!

---
## 5. Vom Sachproblem zur Lösung — Schritt für Schritt

Die folgende Übersicht zeigt den **Fahrplan**, mit dem man jedes Extremwertproblem systematisch löst.

In [ ]:
def methode_visualisieren():
    """Zeigt den Lösungsweg als Flussdiagramm mit einem durchgerechneten Beispiel."""

    fig, (ax_flow, ax_plot) = plt.subplots(1, 2, figsize=(14, 7),
                                            gridspec_kw={'width_ratios': [1, 1.2]})

    # --- Links: Flussdiagramm ---
    ax_flow.set_xlim(0, 10)
    ax_flow.set_ylim(0, 10)
    ax_flow.axis('off')
    ax_flow.set_title('Fahrplan: Extremwertaufgabe', fontsize=14, fontweight='bold')

    schritte = [
        (5, 9.0, "1. Skizze anfertigen\nVariablen einzeichnen",  '#E8F4FD'),
        (5, 7.5, "2. Zielfunktion aufstellen\n$Z = f(x, y)$ — Was wird optimiert?", '#FFF3E0'),
        (5, 6.0, "3. Nebenbedingung aufstellen\n$g(x, y) = c$ — Was ist fest?", '#FFF3E0'),
        (5, 4.5, "4. Einsetzen: $y$ eliminieren\n$Z(x) = ...$ — nur noch eine Variable", '#E8F5E9'),
        (5, 3.0, "5. Ableiten: $Z'(x) = 0$\nKritische Stelle finden", '#E8F5E9'),
        (5, 1.5, "6. $Z''(x) > 0$ → Min\n$Z''(x) < 0$ → Max\n+ Ränder prüfen!", '#FCE4EC'),
    ]

    for x_pos, y_pos, text, color in schritte:
        box = patches.FancyBboxPatch((1.2, y_pos - 0.55), 7.6, 1.1,
                                      boxstyle="round,pad=0.15",
                                      facecolor=color, edgecolor='#555', linewidth=1.5)
        ax_flow.add_patch(box)
        ax_flow.text(x_pos, y_pos, text, ha='center', va='center',
                    fontsize=10, family='sans-serif')

    # Pfeile zwischen den Boxen
    for i in range(len(schritte) - 1):
        ax_flow.annotate('', xy=(5, schritte[i+1][1] + 0.55),
                         xytext=(5, schritte[i][1] - 0.55),
                         arrowprops=dict(arrowstyle='->', color='#555', lw=2))

    # --- Rechts: Durchgerechnetes Beispiel (Blech-Quader) ---
    ax_plot.set_title('Beispiel: Offener Quader aus Blech (20 cm)',
                      fontsize=13, fontweight='bold')

    a = 20  # Seitenlänge Blech
    xs = np.linspace(0, a/2, 300)
    Vs = xs * (a - 2*xs)**2

    x_opt = a / 6
    V_opt = x_opt * (a - 2*x_opt)**2

    ax_plot.plot(xs, Vs, 'b-', linewidth=2.5,
                 label=r'$V(x) = x \cdot (20-2x)^2$')
    ax_plot.plot(x_opt, V_opt, 'r*', markersize=20, zorder=5,
                 label=rf'Maximum: $x = \frac{{10}}{{3}} \approx {x_opt:.2f}$ cm')
    ax_plot.axvline(x=x_opt, color='red', linestyle=':', alpha=0.5)

    # Schritte als Annotationen
    ax_plot.annotate(
        f'$V_{{max}} \\approx {V_opt:.1f}$ cm³',
        xy=(x_opt, V_opt), xytext=(x_opt + 2, V_opt + 30),
        fontsize=12, arrowprops=dict(arrowstyle='->', color='red', lw=1.5),
        bbox=dict(boxstyle='round', facecolor='#FCE4EC', alpha=0.9)
    )

    ax_plot.set_xlabel('Ausschnitt $x$ [cm]', fontsize=12)
    ax_plot.set_ylabel('Volumen $V$ [cm³]', fontsize=12)
    ax_plot.set_xlim(0, 10)
    ax_plot.set_ylim(0, 700)
    ax_plot.grid(True, alpha=0.3)
    ax_plot.legend(fontsize=11, loc='upper right', framealpha=0.9)

    plt.tight_layout()
    plt.show()

methode_visualisieren()

### Zusammenfassung: Steckbriefaufgaben — Fahrplan

| Schritt | Was tun? |
|---------|----------|
| 1 | **Ansatz:** Allgemeines Polynom $n$-ten Grades aufschreiben: $f(x) = a_n x^n + \ldots + a_1 x + a_0$ |
| 2 | **Bedingungen übersetzen:** $f(x_0) = y_0$, $f'(x_0) = m$, $f''(x_0) = 0$, ... |
| 3 | **Symmetrie nutzen:** Achsensymmetrie → nur gerade Exponenten, Punktsymmetrie → nur ungerade |
| 4 | **LGS aufstellen:** Bedingungen einsetzen → $n+1$ Gleichungen für $n+1$ Unbekannte |
| 5 | **LGS lösen** (Gauß, Einsetzung, oder CAS) |
| 6 | **Probe:** Alle Bedingungen in die Lösung einsetzen |